# lenia-vlm on Colab

Runs the Lenia simulator on a GPU (PyTorch) with results saved to your Google Drive.

**Before the first run**
1. *Runtime → Change runtime type → GPU* (a T4 is fine).
2. *Secrets* (the key icon in the left sidebar): add `GITHUB_TOKEN`, a fine-grained GitHub token with
   read-only *Contents* access to `feli10/lenia-vlm`, and turn on notebook access. Only needed while
   the repo is private.
3. Optional, only for VLM judging: add `OPENROUTER_API_KEY` and `TYPESAFE_API_KEY` the same way.
   They are written to a `.env` on this temporary machine and never printed.

Run the cells top to bottom. Re-running the setup cells is harmless (they pull the latest code).

## 1. GPU and Drive

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> GPU"
import os
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/lenia-runs'      # every result goes here and survives disconnects
os.makedirs(RUNS, exist_ok=True)
print('results folder:', RUNS)

## 2. Code, dependencies, data

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

REPO, DIR = 'github.com/feli10/lenia-vlm.git', '/content/lenia-vlm'
tok = secret('GITHUB_TOKEN')
url = f'https://x-access-token:{tok}@{REPO}' if tok else f'https://{REPO}'
if not os.path.isdir(DIR):
    r = subprocess.run(['git', 'clone', '-q', url, DIR], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', DIR, 'pull', '-q', url, 'main'], capture_output=True, text=True)
print('git:', 'ok' if r.returncode == 0 else r.stderr.replace(tok or '\0', '***'))
subprocess.run(['git', '-C', DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'])   # don't keep the token in .git
os.chdir(DIR)
!git log --oneline -1
!pip install -q -e .

In [ ]:
# Chan's catalogue (MIT licence), and API keys for the judges if you added them as secrets
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/animals.json'):
    !curl -sL -o data/animals.json https://raw.githubusercontent.com/Chakazul/Lenia/master/Python/animals.json
print('catalogue:', os.path.getsize('data/animals.json'), 'bytes')
keys = {k: secret(k) for k in ('OPENROUTER_API_KEY', 'TYPESAFE_API_KEY')}
with open('.env', 'w') as f:
    for k, v in keys.items():
        if v:
            f.write(f'{k}={v}\n')
print('judge keys present:', [k for k, v in keys.items() if v] or 'none (fine for simulation-only runs)')

## 3. Check the GPU simulator, then time it

Compares the PyTorch step with the numpy reference from identical starts, then times both. Do not
use the GPU for runs unless this ends with **ALL CHECKS PASSED**.

In [ ]:
!python -m lenia --backend auto bench

Optional: the full test suite (about a minute).

In [ ]:
!pip install -q pytest && python -m pytest -q tests

## 4. A first GPU run

A small multikernel survey at double resolution, straight to Drive. Compare its time with the same
command on your Mac to see the real end-to-end speedup (the bench above times the update step only;
the alive classifier still runs on the CPU).

In [ ]:
import time
t0 = time.time()
!python -m lenia --backend auto --runs {RUNS} --config configs/r26.yaml survey --mode multikernel --n 64 --out {RUNS}/gpu-test-survey
print(f'{time.time() - t0:.0f} s')

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{RUNS}/gpu-test-survey/sheet_alive.png'))[:1]:
    display(Image(p, width=700))
!cat {RUNS}/gpu-test-survey/summary.json | head -20

## 5. Experiment A: multi-channel creatures (run 2, alive v2)

Starts from Chan's multi-channel `found/` collections (files 221, 223, 231, 231b, 232, 233, 233c, 233s;
about 2,450 creatures), all brought to kernel radius 20 in a 256x256 world (`configs/found.yaml`).

1. **Download** the collections (small; re-run after a disconnect, the machine is wiped).
2. **Survey**: run every creature from its own pattern, keep the living ones, then calibrate the checks
   on them: each living creature is also run at 2x resolution, and its texture is measured
   (`calibration.json`, `sheet_texture.png`). All creatures, roughly 20-40 minutes on a T4.
3. **Search**: novelty search from 64 diverse living creatures that pass the calibrated checks, with
   the same hard checks for every child (alive at 100, still alive at 200, texture, coherent at 2x).
   It saves a checkpoint to Drive every 2 minutes: after a disconnect, re-run sections 1-2 and the
   same cell, and it continues. About 35-50 minutes.

Run 1 (each creature at Chan's own R, checks calibrated on the classic catalogue) is in
`lenia-runs/found-survey` and `lenia-runs/expA-explore`; this run writes new folders.


In [ ]:
!python -m lenia found download

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml found survey --out {RUNS}/found-survey-v2

In [ ]:
from IPython.display import Image, display
import glob, json
s = json.load(open(f'{RUNS}/found-survey-v2/summary.json'))
print(json.dumps({n: f"{f['alive']}/{f['creatures']}" for n, f in s['per_file'].items()}, indent=1))
print(json.dumps(s['calibration'], indent=1))
display(Image(f'{RUNS}/found-survey-v2/sheet_texture.png', width=700))
for p in sorted(glob.glob(f'{RUNS}/found-survey-v2/sheet_23*.png'))[:2]:
    display(Image(p, width=700))

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml explore --from-found {RUNS}/found-survey-v2 --roots 64 --evals 3200 --out {RUNS}/expA2-explore

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml explore --replot {RUNS}/expA2-explore --no-catalogue
for name in ('sheet_novel.png', 'sheet_lineages.png'):
    display(Image(f'{RUNS}/expA2-explore/{name}', width=700))

## 6. Phase 2: the labelling pool

Picks about 300 of Chan's living creatures (those passing the calibrated checks) and up to 10
descendants per lineage from the run-2 search, re-simulates each, renders the fixed-camera image the
judges see, holds out 20% of the creatures for testing, and makes the pairs. A few minutes on a T4.

The labelling itself (Luna, Jev) runs on your Mac, where the API keys are: see `docs/lenia.md`,
"Phase 2: labelling pool". Your Mac reads the pool straight from Drive.


In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml pool build

In [ ]:
from IPython.display import Image, display
import json
print(json.dumps(json.load(open(f'{RUNS}/pool/build.json')), indent=1)[:800])
for name in ('sheet_chan.png', 'sheet_search.png'):
    display(Image(f'{RUNS}/pool/{name}', width=700))

## 7. Phase 3: embed the pool for the student

Runs a frozen DINOv2 image model over every creature image of the labelling pool (each two-view image
is split into its motion view and its close-ups). A minute or two on a T4. Training the student itself
is numpy only (`lenia distill train`) and runs anywhere, including your Mac.


In [ ]:
!python -m lenia --backend auto --runs {RUNS} distill embed --pool {RUNS}/pool2 --model dinov2_vits14

## 8. Phase 4: search against the student

Two runs from the same 16 founders (pool creatures spread across the student's score range), 60
generations, 3 mutants per lineage per generation, under the frozen alive check:
- `--reward student`: the distilled image reward (needs the student in `pool2/distill_dinov2_vits14_curate_dual`)
- `--reward features`: the five-statistic reward fitted on the same labels, for comparison

No API calls: the teacher checks the results afterwards on your Mac (`lenia rsearch check`). Both runs
resume after a disconnect (re-run sections 1-2, then the same cell).


In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml rsearch run --reward student

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml rsearch run --reward features

In [ ]:
from IPython.display import Image, display
for r in ('student', 'features'):
    display(Image(f'{RUNS}/rsearch-{r}/lineages.png', width=900))

## 9. Phase 5: MAP-Elites against the student

A 10 x 10 map over size x speed (the teacher's first two preferences), seeded with every pool creature.
Each iteration mutates 64 elites drawn from the map, runs the frozen alive check, and a viable mutant
takes its size/speed cell if it scores higher than the elite there. 80 iterations = 5,120 mutants,
roughly 3 hours per run on a T4 (about 2 seconds per mutant). Both runs resume after a disconnect
(re-run sections 1-2, then the same cell); `--iterations 120` continues a finished run.

Run the student first; the features run is the comparison and can wait for another session.


In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml qd run --reward student

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml qd run --reward features

In [ ]:
from IPython.display import Image, display
for r in ('student', 'features'):
    for name in ('map.png', 'progress.png', 'elites.png'):
        try:
            display(Image(f'{RUNS}/qd-{r}/{name}', width=900))
        except FileNotFoundError:
            pass

## 10. Mutation-operator pilot

Which variation operators reach genuinely new creatures and stay viable? 64 final elites from the two
MAP-Elites maps (32 each) get one child per operator: `copy` (baseline: the parent unchanged), the current
mutation at 1x/2x/4x, pattern operators (`reseed`, `cut`, `merge`: same rule, new start), structural
edits (`add`, `remove`, `duplicate` a kernel, `rewire` a channel; each with a short m/s re-tune), and
`crossover` between parents with the same layout. Each child goes through the frozen alive check (alive
at 100, coherent and keeping texture at 2x, alive at 200); viable ones are rendered, embedded with the
student's DINOv2 and measured against their parent and the pool.

Decision rule (written before running): an operator passes if at least 10% of its children are viable and
the median viable child is beyond 0.5 from the pool. No API calls. Roughly 30-45 minutes on a T4 (12
operators x 64 children, plus the re-tune tries); results are saved after each operator, so after a
disconnect re-run sections 1-2 and the same cell. `--operators mut4,crossover` runs a subset.

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml pilot run

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{RUNS}/pilot-operators/report.md').read()))
display(Image(f'{RUNS}/pilot-operators/sheet.png', width=900))

## 11. Catalogue rankings: student vs teacher

What do the student and the teacher consider most worth including? Two sets, each rendered exactly as the
pool (two-view, R = 20):
- **found**: all 644 living creatures of the found/ survey; 257 are in the pool (most in its training
  split), 387 the student has never seen
- **classic**: Chan's classic catalogue (one channel, one kernel; about 490 fit at R = 20), drawn grey on
  black. A different kind of creature, so the student's agreement here is a distribution-shift test, but
  the image style changes with it

This cell renders, embeds and scores everything with the student and makes the pairs for the teacher (a few
minutes on a T4). It ends with student-only rankings. The teacher's labels come from your Mac
(`lenia rank label`, about 2 x 8 x 1,100 = 18,000 calls, around $1-3 for Luna), then `lenia rank report`.

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/found.yaml rank build

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{RUNS}/rank-catalogue/report.md').read()))
for s in ('found', 'classic'):
    for name in (f'rank_{s}_student.png', f'rank_{s}_student_size_out.png'):
        try:
            display(Image(f'{RUNS}/rank-catalogue/{name}', width=900))
        except FileNotFoundError:
            pass

## Notes

- Every command takes `--backend auto --runs {RUNS}`; results then live in your Drive folder.
- The free tier disconnects idle sessions and caps sessions at about 12 hours. `lenia explore` and
  `lenia guided run` resume after a disconnect (re-run sections 1-2, then the same command with the
  same `--out`). After a disconnect the machine is wiped, your Drive is not.
- Claude can read your Drive folder: just say when a run has finished.